# 최종 Test 평가

## 평가 원칙

- Baseline → V1 → V2의 개선 방향과 최종 후보 선택은 **Validation에서 모두 완료**
- V2를 최종 후보로 고정한 뒤 **Test Set은 마지막 일반화 성능 확인에만 사용**
- Test 결과를 보고 추가 학습, 하이퍼파라미터 변경, 모델 재선택을 수행하지 않음
- Baseline/V1/V2 Test 결과는 개선 과정의 사후 확인용 비교이며, V2 선택 근거는 Validation 결과
- Test Set: **733 images / 749 objects**

> 아래 실행은 세 모델에 동일한 `conf=0.25` 조건을 적용한 상대 비교입니다.  
> Ultralytics의 표준 PR 기반 mAP benchmark는 낮은 confidence 기본값을 사용하는 방식이 일반적이므로, 본 표의 수치는 **동일 조건에서의 모델 간 비교**로 해석합니다.

In [1]:
from pathlib import Path

import pandas as pd
from ultralytics import YOLO


ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
DATA_YAML = ROOT / "data" / "yolo_subset" / "data.yaml"

MODELS = {
    "baseline": 640,
    "improved_v1": 960,
    "improved_v2": 960,
}


# Validation에서 모든 개선 의사결정 종료 후 Test 최종 확인
rows = []

for name, img_size in MODELS.items():
    model = YOLO(str(ROOT / "models" / name / "weights" / "best.pt"))

    result = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=img_size,
        conf=0.25,
        verbose=False,
        plots=False,
    )

    p, r = result.box.mp, result.box.mr

    rows.append([
        name, img_size, p, r, 2 * p * r / (p + r),
        result.box.map50, result.box.map75, result.box.map,
        result.speed["inference"],
    ])


columns = [
    "Model", "Image Size", "Precision", "Recall", "F1",
    "mAP50", "mAP75", "mAP50-95", "Inference(ms)"
]

df = pd.DataFrame(rows, columns=columns)

display(df.round(4))

,Model,Image Size,Precision,Recall,F1,mAP50,mAP75,mAP50-95,Inference(ms)
0,baseline,640,0.9483,0.8838,0.9149,0.9143,0.7051,0.6139,2.4686
1,improved_v1,960,0.9661,0.9252,0.9452,0.9405,0.7500,0.6561,3.2162
2,improved_v2,960,0.9704,0.9372,0.9535,0.9419,0.7801,0.6852,2.9315


## 1. Test 성능 비교

| Model | Image Size | Precision | Recall | F1 | mAP50 | mAP75 | mAP50-95 | Inference(ms) |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Baseline | 640 | 0.9483 | 0.8838 | 0.9149 | 0.9143 | 0.7051 | 0.6139 | 2.47 |
| Improved V1 | 960 | 0.9661 | 0.9252 | 0.9452 | 0.9405 | 0.7500 | 0.6561 | 3.22 |
| Improved V2 | 960 | **0.9704** | **0.9372** | **0.9535** | **0.9419** | **0.7801** | **0.6852** | 2.93 |

모든 정확도 지표에서 V2가 가장 높았으며, 이는 **Validation에서 고정한 최종 후보의 성능 향상이 Test에서도 유지되는지 확인한 결과**로 해석

Inference time은 하드웨어와 실행 변동의 영향을 받으므로 정확도 모델 선택 근거가 아니라 참고값으로만 사용

## 2. 최종 판단

### Validation에서 완료된 개선 의사결정

- Baseline: Q1 Recall **0.7489**, FN **123개**, FP **99개**
- V1: Input Size **640 → 960**
  - Q1 Recall **0.7489 → 0.8265**
  - FN **123 → 80개**, FP **99 → 75개**
- V2: Input Size 960 유지, Epoch **30 → 100**
  - 위치 부정확 FN **23 → 10개**
  - FP **75 → 51개**
  - mAP75 **0.7225 → 0.7703**
  - mAP50-95 **0.6202 → 0.6500**

위 Validation 결과를 근거로 **V2를 최종 후보로 먼저 고정**

### 최종 Test 확인

V2 Test: **Precision 0.9704 / Recall 0.9372 / F1 0.9535 / mAP50 0.9419 / mAP75 0.7801 / mAP50-95 0.6852**

Baseline 대비 Test 변화:

- Precision **+0.0221**
- Recall **+0.0534**
- F1 **+0.0386**
- mAP50 **+0.0276**
- mAP75 **+0.0750**
- mAP50-95 **+0.0713**

Validation에서 확인한 **전체 Recall 및 높은 IoU 조건의 성능 개선이 Test에서도 유지됨**

다만 Test에서는 객체 크기별 Recall이나 실패 유형을 다시 분석하지 않았으므로, **극소형 UAV 개선이 Test에서도 동일하게 유지됐다고 별도로 단정하지 않음**

따라서 Test는 V2 선택의 근거가 아니라 **Validation에서 선택한 모델의 최종 일반화 확인**으로 사용

## 3. 잔여 오류 및 향후 개선 방향

Test Set은 최종 확인용으로 유지하기 위해 추가 실패 분석이나 threshold 튜닝에 사용하지 않음

아래 잔여 오류는 **V2 Validation 실패 분석 결과**를 기준으로 정리

| 잔여 문제 | Validation 확인 결과 | 향후 개선 방향 |
|---|---|---|
| 극소형 UAV 미탐 | Q1 Recall 0.8311로 다른 크기 구간보다 낮음 | Q1 학습 샘플 보강 및 작은 객체 중심 데이터 증강 |
| 완전 미검출 | FN 76개 중 미검출 41개 | 수목·복잡한 배경·낮은 대비의 어려운 사례 추가 학습 |
| 낮은 신뢰도 | 낮은 신뢰도 FN 25개 | Validation PR 결과로 운영 confidence threshold 재검토 |
| 배경 오탐 | 조류·비행기·건물 구조물·바위·그림자 등과 혼동 | 실제 FP 사례를 Hard Negative로 추가 |
| bbox 위치 오차 | 위치 부정확 FN 10개 잔존 | annotation 일관성 점검과 극소형 bbox 정밀도 개선 |
| 데이터 일반화 | 같은 공개 데이터셋 내부 split에서 검증 | 다른 촬영 환경의 외부 데이터로 추가 일반화 검증 필요 |

### 개선 우선순위

1. **Hard Negative 보강**: 실제 FP로 확인된 배경·유사 물체를 학습에 반영
2. **극소형 UAV 데이터 보강**: Q1 및 복잡한 배경의 미검출 사례 중심
3. **Confidence Threshold 재검토**: Validation에서 Precision-Recall 균형으로 운영 threshold 선택
4. **독립 환경 일반화 검증**: 현재 데이터셋과 다른 촬영 환경에서 최종 V2 추가 확인

추가 Epoch만 늘리는 것보다 **잔여 실패 사례를 직접 데이터에 반영하는 개선의 우선순위가 높음**

### 최종 결론

**Baseline 실패 분석 → Input Size 증가(V1) → Validation 개선 확인 → Epoch 증가(V2) → Validation에서 최종 후보 고정 → Test 1회 최종 확인**

과제 요구의 핵심인 **문제 정의, 실패 원인 분석, 근거 기반 개선, 개선 전후 정량 비교, 한계 인식**이 하나의 흐름으로 연결됨